## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Upload and Extract Dataset

In [ ]:
from google.colab import files
import zipfile
import os

uploaded = files.upload()

zip_file = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall("fraud_dataset")

## 3. Load the Dataset

In [ ]:
df = pd.read_csv("fraud_dataset/creditcard.csv")

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

## 4. Dataset Information

In [ ]:
print("Dataset Information:")
df.info()

print("\n" + "="*50)

print("Column Names:")
print(df.columns.tolist())

print("\n" + "="*50)

print("Missing Values:")
print(df.isnull().sum())

print("\n" + "="*50)

print("Duplicate Rows:", df.duplicated().sum())

## 5. Class Distribution and Imbalance Analysis

In [ ]:
class_counts = df["Class"].value_counts()

print("Class Distribution:")
print(class_counts)

print("\nClass Percentage:")
print((class_counts / len(df) * 100).round(4))

## 5.1 Fraud vs Non-Fraud Transactions

In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(x="Class", data=df)

plt.title("Fraud vs Non-Fraud Transactions")
plt.xlabel("Class (0 = Non-Fraud, 1 = Fraud)")
plt.ylabel("Number of Transactions")

plt.show()

## 6. Fraudulent Transaction Percentage

In [ ]:
fraud_count = df["Class"].sum()
total_transactions = len(df)

fraud_percentage = (fraud_count / total_transactions) * 100
non_fraud_percentage = 100 - fraud_percentage

print("Total Transactions:", total_transactions)
print("Fraudulent Transactions:", fraud_count)
print("Non-Fraudulent Transactions:", total_transactions - fraud_count)

print("\nFraud Percentage:", round(fraud_percentage, 4), "%")
print("Non-Fraud Percentage:", round(non_fraud_percentage, 4), "%")

## 7. Transaction Amount Distribution

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=df,
    x="Amount",
    hue="Class",
    bins=50,
    kde=True,
    element="step"
)

plt.title("Transaction Amount Distribution: Fraud vs Non-Fraud")
plt.xlabel("Transaction Amount")
plt.ylabel("Number of Transactions")

plt.show()

## 8. Time-of-Day Analysis

In [ ]:
df["Hour"] = (df["Time"] / 3600).astype(int)

hourly_fraud = df.groupby("Hour")["Class"].sum()

plt.figure(figsize=(12, 5))

sns.lineplot(x=hourly_fraud.index, y=hourly_fraud.values, marker="o")

plt.title("Fraudulent Transactions by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Number of Fraudulent Transactions")
plt.xticks(range(0, 24))

plt.show()

## 9. Why Accuracy Is a Misleading Metric

Fraud detection datasets are highly imbalanced because fraudulent transactions represent only a very small percentage of all transactions. In this dataset, only a small fraction of transactions are fraudulent.

Therefore, accuracy can be a misleading metric. A model could predict almost every transaction as non-fraud and still achieve very high accuracy while failing to detect actual fraud cases.

For fraud detection, Precision, Recall, F1-Score, and AUC-ROC are more informative evaluation metrics. Recall is particularly important when the main objective is to identify as many fraudulent transactions as possible.

## 10. Feature and Target Preparation

In [ ]:
X = df.drop("Class", axis=1)
y = df["Class"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

## 11. Stratified Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nFraud cases in training set:", y_train.sum())
print("Fraud cases in testing set:", y_test.sum())

## 12. Handling Class Imbalance Using SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before SMOTE:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(y_train_smote.value_counts())

## 13. Logistic Regression Model

In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train_smote, y_train_smote)

y_pred_lr = logistic_model.predict(X_test)
y_prob_lr = logistic_model.predict_proba(X_test)[:, 1]

print("Logistic Regression model trained successfully!")

## 14. Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_smote, y_train_smote)

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest model trained successfully!")

## 15. Model Evaluation

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

lr_precision = precision_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)
lr_auc = roc_auc_score(y_test, y_prob_lr)

rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)
rf_auc = roc_auc_score(y_test, y_prob_rf)

results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Precision": [lr_precision, rf_precision],
    "Recall": [lr_recall, rf_recall],
    "F1-Score": [lr_f1, rf_f1],
    "AUC-ROC": [lr_auc, rf_auc]
})

results

## 16. AUC-ROC Curve

In [ ]:
from sklearn.metrics import roc_curve

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

plt.figure(figsize=(8, 6))

plt.plot(fpr_lr, tpr_lr, label=f"Logistic Regression (AUC = {lr_auc:.3f})")
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {rf_auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")

plt.title("AUC-ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()

plt.show()

## 17. Precision vs Recall in Fraud Detection

Recall is especially important in fraud detection because missing a fraudulent transaction can result in financial loss.

Precision is also important because too many false fraud alerts can inconvenience genuine customers.

Therefore, fraud detection requires a balance between Recall and Precision. In many real-world fraud detection systems, higher Recall is preferred when the cost of missing fraud is higher than the cost of investigating false alerts.

## 18. Feature Importance - Random Forest

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

print(feature_importance.head(15))

plt.figure(figsize=(10, 6))

sns.barplot(
    data=feature_importance.head(15),
    x="Importance",
    y="Feature"
)

plt.title("Top 15 Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")

plt.show()

## 19. Scalability for 1 Million Transactions per Hour

If the fraud detection system needs to process 1 million transactions per hour, the model should be deployed as a scalable real-time prediction service.

The system could use distributed processing, batch or streaming pipelines, parallel model inference, and cloud-based infrastructure. A trained model can be deployed behind an API so transactions can be evaluated automatically.

For large-scale production systems, the data pipeline should also monitor model performance, processing latency, class imbalance, and changes in fraud patterns.

## 20. Business Recommendations

1. Prioritize Recall because undetected fraudulent transactions can cause financial losses.

2. Monitor Precision to reduce unnecessary fraud alerts for genuine customers.

3. Use SMOTE or other imbalance-handling techniques during model training because fraudulent transactions are extremely rare.

4. Random Forest can be considered when it provides better overall Precision, Recall, F1-Score, and AUC-ROC performance.

5. Continuously retrain and monitor the model because fraud patterns can change over time.

6. For large transaction volumes, use scalable and distributed infrastructure for real-time fraud detection.

## 21. Conclusion

This project developed a machine learning pipeline for detecting fraudulent credit card transactions from a highly imbalanced dataset.

The analysis showed that fraudulent transactions represent only a very small percentage of the total transactions, making accuracy an unreliable evaluation metric.

SMOTE was applied to address class imbalance, and Logistic Regression and Random Forest models were trained and evaluated using Precision, Recall, F1-Score, and AUC-ROC.

The results can help identify the better-performing model for fraud detection while considering the trade-off between detecting fraudulent transactions and avoiding false alerts.